# Notebook 01 — Baseline Model Training (Part I)

**Objective:** Train YOLOv11s on VisDrone as the baseline and record all standard metrics.

**Model:** `yolo11s` — small variant, good balance of speed and accuracy for a baseline.  
**Epochs:** 50 (reasonable for a CMPE 401 project timeline)  
**Image size:** 640 (standard)  

All metrics are saved under `results/baseline/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO

sns.set_theme(style='whitegrid')
print('ROOT:', ROOT)

## 1. Configuration

In [ ]:
# ──────────────── BASELINE CONFIGURATION ────────────────
MODEL_CHECKPOINT = 'yolo11s.pt'          # COCO-pretrained starting point
DATA_CFG         = ROOT / 'configs' / 'visdrone.yaml'
EXP_NAME         = 'baseline'
PROJECT_DIR      = ROOT / 'results'

EPOCHS   = 50
IMGSZ    = 640
BATCH    = 16     # adjust to your GPU memory (8 if <8 GB VRAM)
LR0      = 0.01
LRF      = 0.01   # final LR = LR0 * LRF
DEVICE   = '0'    # '0' for first GPU, 'cpu' for CPU-only
WORKERS  = 4
PATIENCE = 20     # early stopping

print(f'Config loaded. Data: {DATA_CFG}')

## 2. Train Baseline

In [ ]:
model = YOLO(MODEL_CHECKPOINT)

results = model.train(
    data     = str(DATA_CFG),
    epochs   = EPOCHS,
    imgsz    = IMGSZ,
    batch    = BATCH,
    lr0      = LR0,
    lrf      = LRF,
    patience = PATIENCE,
    device   = DEVICE,
    workers  = WORKERS,
    project  = str(PROJECT_DIR),
    name     = EXP_NAME,
    exist_ok = True,
    plots    = True,
    verbose  = True,
)

BEST_WEIGHTS = Path(results.save_dir) / 'weights' / 'best.pt'
print(f'\nBest weights: {BEST_WEIGHTS}')

## 3. Load Training Metrics CSV

In [ ]:
run_dir = PROJECT_DIR / EXP_NAME
csv_path = run_dir / 'results.csv'

df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()   # strip whitespace from column names
print('Columns:', df.columns.tolist())
print(df.tail(5).to_string())

## 4. Training & Validation Loss Curves

In [ ]:
def get_col(df, *candidates):
    """Return the first matching column name."""
    for c in candidates:
        if c in df.columns:
            return c
    return None

# Ultralytics column naming varies by version
epoch_col    = get_col(df, 'epoch', 'Epoch')
train_box    = get_col(df, 'train/box_loss', 'train/box_om')
train_cls    = get_col(df, 'train/cls_loss')
train_dfl    = get_col(df, 'train/dfl_loss')
val_box      = get_col(df, 'val/box_loss', 'val/box_om')
val_cls      = get_col(df, 'val/cls_loss')
val_dfl      = get_col(df, 'val/dfl_loss')
map50_col    = get_col(df, 'metrics/mAP50(B)', 'metrics/mAP_0.5')
map95_col    = get_col(df, 'metrics/mAP50-95(B)', 'metrics/mAP_0.5:0.95')
prec_col     = get_col(df, 'metrics/precision(B)', 'metrics/precision')
rec_col      = get_col(df, 'metrics/recall(B)', 'metrics/recall')

epochs = df[epoch_col] if epoch_col else range(len(df))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# --- Box loss ---
if train_box and val_box:
    axes[0,0].plot(epochs, df[train_box], label='Train box loss', color='royalblue')
    axes[0,0].plot(epochs, df[val_box],   label='Val box loss',   color='tomato', linestyle='--')
    axes[0,0].set_title('Box Loss')
    axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Loss')
    axes[0,0].legend()

# --- Cls loss ---
if train_cls and val_cls:
    axes[0,1].plot(epochs, df[train_cls], label='Train cls loss', color='royalblue')
    axes[0,1].plot(epochs, df[val_cls],   label='Val cls loss',   color='tomato', linestyle='--')
    axes[0,1].set_title('Classification Loss')
    axes[0,1].set_xlabel('Epoch'); axes[0,1].set_ylabel('Loss')
    axes[0,1].legend()

# --- DFL loss ---
if train_dfl and val_dfl:
    axes[0,2].plot(epochs, df[train_dfl], label='Train DFL loss', color='royalblue')
    axes[0,2].plot(epochs, df[val_dfl],   label='Val DFL loss',   color='tomato', linestyle='--')
    axes[0,2].set_title('DFL Loss')
    axes[0,2].set_xlabel('Epoch'); axes[0,2].set_ylabel('Loss')
    axes[0,2].legend()

# --- mAP50 ---
if map50_col:
    axes[1,0].plot(epochs, df[map50_col], color='green')
    axes[1,0].set_title('mAP@50')
    axes[1,0].set_xlabel('Epoch'); axes[1,0].set_ylabel('mAP50')

# --- mAP50-95 ---
if map95_col:
    axes[1,1].plot(epochs, df[map95_col], color='darkorange')
    axes[1,1].set_title('mAP@50-95')
    axes[1,1].set_xlabel('Epoch'); axes[1,1].set_ylabel('mAP50-95')

# --- Precision & Recall ---
if prec_col and rec_col:
    axes[1,2].plot(epochs, df[prec_col], label='Precision', color='royalblue')
    axes[1,2].plot(epochs, df[rec_col],  label='Recall',    color='tomato', linestyle='--')
    axes[1,2].set_title('Precision & Recall')
    axes[1,2].set_xlabel('Epoch'); axes[1,2].set_ylabel('Value')
    axes[1,2].legend()

plt.suptitle(f'Baseline Training — {MODEL_CHECKPOINT} | VisDrone | {EPOCHS} epochs', fontsize=13)
plt.tight_layout()
plt.savefig(run_dir / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Final Validation Metrics

In [ ]:
val_results = model.val(
    data    = str(DATA_CFG),
    split   = 'val',
    imgsz   = IMGSZ,
    device  = DEVICE,
    plots   = True,
    verbose = True,
)

rd = val_results.results_dict
baseline_metrics = {
    'model':     MODEL_CHECKPOINT,
    'mAP50':     round(float(rd.get('metrics/mAP50(B)', 0)), 4),
    'mAP50_95':  round(float(rd.get('metrics/mAP50-95(B)', 0)), 4),
    'precision': round(float(rd.get('metrics/precision(B)', 0)), 4),
    'recall':    round(float(rd.get('metrics/recall(B)', 0)), 4),
}

print('\nBaseline Metrics')
print('=' * 35)
for k, v in baseline_metrics.items():
    print(f'  {k:<12}: {v}')

# Save for later comparison
with open(run_dir / 'baseline_metrics.json', 'w') as f:
    json.dump(baseline_metrics, f, indent=2)
print(f'\nSaved to {run_dir / "baseline_metrics.json"}')

## 6. Per-Class Metrics

In [ ]:
CLASS_NAMES = ['pedestrian', 'people', 'bicycle', 'car', 'van',
               'truck', 'tricycle', 'awning-tricycle', 'bus', 'motor']

# Per-class AP values are in val_results.box.ap_class_index and .ap
try:
    ap50_per_class = val_results.box.ap50
    names = [CLASS_NAMES[i] if i < len(CLASS_NAMES) else str(i)
             for i in val_results.box.ap_class_index]

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.barh(names, ap50_per_class, color=plt.cm.tab10.colors[:len(names)])
    ax.set_xlabel('AP@50')
    ax.set_title('Per-Class AP@50 — Baseline')
    for bar, v in zip(bars, ap50_per_class):
        ax.text(v + 0.005, bar.get_y() + bar.get_height()/2, f'{v:.3f}', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(run_dir / 'per_class_ap50.png', dpi=150, bbox_inches='tight')
    plt.show()
except AttributeError:
    print('Per-class AP not available in this Ultralytics version.')

## 7. Sample Predictions

In [ ]:
import random

DATA_ROOT = ROOT / 'Data'
val_imgs = sorted((DATA_ROOT / 'VisDrone2019-DET-val' / 'images').glob('*.jpg'))
sample_imgs = random.sample(val_imgs, min(4, len(val_imgs)))

pred_model = YOLO(str(BEST_WEIGHTS))
preds = pred_model(sample_imgs, imgsz=IMGSZ, conf=0.25, device=DEVICE)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, result in zip(axes, preds):
    ax.imshow(result.plot()[:, :, ::-1])  # BGR→RGB
    ax.axis('off')
    ax.set_title(Path(result.path).stem[:20], fontsize=8)

plt.suptitle('Baseline — Sample Predictions on Val Set', fontsize=12)
plt.tight_layout()
plt.savefig(run_dir / 'sample_predictions.png', dpi=120, bbox_inches='tight')
plt.show()

## Summary

| Metric | Value |
|--------|-------|
| Model  | yolo11s (COCO pretrained → VisDrone fine-tuned) |
| mAP@50 | *see output above* |
| mAP@50-95 | *see output above* |
| Precision | *see output above* |
| Recall | *see output above* |

The baseline provides the reference point for all subsequent experiments.  
Continue to **Notebook 02** for loss curve analysis and fitting diagnostics.